# Machine Learning in Drug Discovery
## Part 3a — AlphaFold and Protein Structure Prediction

**Context**: This notebook is the first of two technical demonstrations accompanying a presentation
on *Machine Learning in Drug Discovery* (based on Dara et al., 2022, *Artificial Intelligence Review*).
It focuses on **AlphaFold 2** as a case study of deep learning solving a foundational bottleneck
in the drug discovery pipeline: predicting protein 3D structure from amino acid sequence.

**What this notebook does**:
1. Presents the architecture and key innovations of AlphaFold 2
2. Retrieves real structures from the public AlphaFold Protein Structure Database
3. Parses and analyses per-residue confidence (pLDDT) scores from the PDB files
4. Compares a partially disordered protein (p53) with a compact globular protein (lysozyme)
5. Visualises both structures in interactive 3D, coloured by confidence

**Prerequisites**: Run the first cell to install dependencies, then execute cells in order.

In [ ]:
# Install required packages.
# py3Dmol  : JavaScript-based 3D molecular viewer embedded in Jupyter
# requests : HTTP calls to the AlphaFold DB REST API
# pandas, numpy, matplotlib : standard scientific stack
!pip install py3Dmol requests pandas matplotlib numpy -q

### Imports

All dependencies grouped by category.
No heavy bioinformatics libraries (Biopython, RDKit) are used —
the PDB format is parsed from scratch to keep the code transparent and self-contained.

In [ ]:
# --- Standard library ---
import json
import re

# --- Data manipulation ---
import pandas as pd
import numpy as np

# --- HTTP requests ---
import requests

# --- Visualisation ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- 3D molecular visualisation ---
import py3Dmol
from IPython.display import display

print('All libraries loaded successfully.')

---
## Section 1 — AlphaFold: Architecture and Key Innovations

The **protein folding problem** asks a deceptively simple question: given a linear sequence of amino
acids, what three-dimensional structure does the resulting protein adopt? This structure determines
the protein's function — and thus its role as a drug target. Classical computational methods
struggled with this problem for over 50 years.

**AlphaFold 2** (Jumper et al., *Nature*, 2021) solved it by rethinking both the input data
and the neural architecture.

### Core Architectural Ideas

**Multiple Sequence Alignment (MSA)**: Rather than examining a single protein sequence,
AlphaFold collects hundreds of homologous sequences from across the tree of life.
The key insight: **co-evolved residues tend to be spatially close**.
If mutating residue *i* in one species is consistently compensated by a mutation at residue *j*
in another, those two residues likely contact each other in the folded 3D structure.
Evolution thereby encodes millions of years of structural constraints as correlated sequence variation.

**Evoformer**: The architectural heart of AlphaFold 2. It is a stack of 48 transformer-like
attention blocks that jointly processes two coupled representations:
- A **sequence representation** (the MSA matrix): captures which residues co-vary across species
- A **pairwise representation**: explicitly encodes geometric relationships between every pair of residues

These two tracks continuously exchange information across all 48 layers, allowing the model
to reconcile evolutionary signals with geometric consistency constraints.

**Structure Module**: Takes the Evoformer output and produces full 3D atomic coordinates.
It uses **Invariant Point Attention (IPA)** — an attention mechanism that is equivariant to
rotations and translations, ensuring the predicted structure is physically consistent
regardless of how the coordinate frame is chosen.

### CASP14 Performance

In the 2020 CASP14 blind prediction competition, AlphaFold 2 achieved a **median GDT_TS score
of approximately 92.4** on free-modelling targets — comparable to the accuracy of models
derived directly from experimental crystal structures (~90 GDT_TS).
The benchmark that had stood for 50 years was effectively closed.
This work was recognised with the **2024 Nobel Prize in Chemistry** (jointly with David Baker).

### The pLDDT Confidence Score

A critical output of AlphaFold is not just *what* structure it predicts, but *how confident it is*:

**pLDDT** (*predicted Local Distance Difference Test*) is a **per-residue confidence score**
ranging from 0 to 100, output by the model as part of its prediction.
It estimates how accurate each residue's predicted position is likely to be —
a self-assessment computed during inference, not an external validation metric.

| pLDDT range | Interpretation |
|:-----------|:---------------|
| > 90       | Very high confidence — reliable for structure-based drug design |
| 70 – 90    | Confident — generally reliable |
| 50 – 70    | Low confidence — treat with caution |
| < 50       | Very low — region likely **intrinsically disordered** |

Crucially: low pLDDT does not mean the model failed.
It means the model correctly identifies that this region has **no single stable conformation**.
AlphaFold does not just predict structure — **it signals when structure prediction is not the right frame**.

> *Reference: Jumper, J. et al. (2021). Highly accurate protein structure prediction with AlphaFold.
> Nature, 596, 583–589. https://doi.org/10.1038/s41586-021-03819-2*

---
## Section 2 — Why This Matters for Drug Discovery

### The Structure Bottleneck

**Target identification** — determining which protein to block, activate, or modulate with a drug —
requires understanding that protein's 3D structure at atomic resolution.
Without a structure, rational drug design is largely blind: we cannot identify binding pockets,
predict which molecules will fit, explain why a compound works, or design better analogues.

Before AlphaFold, experimental structure determination (X-ray crystallography, NMR, Cryo-EM)
had solved structures for roughly **17% of the human proteome**.
The remaining 83% formed a vast structurally dark space, inaccessible to structure-based drug design.

The **AlphaFold Protein Structure Database** (AlphaFold DB), launched in 2021 by DeepMind
and the EMBL-EBI, changed this fundamentally.
By 2023, it contained **over 200 million protein structures**, covering virtually the entire
UniProt database — from bacteria and parasites to the full human proteome.

### Concrete Impact

- **Neglected tropical diseases**: Proteins from the parasites causing malaria, sleeping sickness,
  and Chagas disease now have structural models, enabling rational drug design for diseases
  historically underfunded by pharmaceutical companies.
- **Antibiotic resistance**: Structural models of resistance-conferring enzymes (beta-lactamases,
  efflux pumps, ribosome-modifying enzymes) guide the design of inhibitors that circumvent
  resistance mechanisms.
- **Cancer biology**: Oncoproteins and tumour suppressors whose structures were previously unknown
  can now be examined computationally for druggable pockets.

The database is freely accessible at https://alphafold.ebi.ac.uk.

### An Honest Limitation

AlphaFold predicts **static, apo (unbound) structures** — the conformation a protein adopts
in isolation, without a bound ligand or partner protein.
Drug discovery additionally requires:

- **Conformational dynamics**: many proteins change shape upon binding a ligand or partner
- **Binding pocket prediction**: identifying *where* on the surface a drug should bind
- **Protein–ligand interaction modelling**: predicting binding affinity and binding pose

These remain active research frontiers.
AlphaFold is a powerful and transformative starting point — not a complete solution.
Active work integrates AlphaFold structures with molecular dynamics simulation,
docking engines, and generative models to address the full drug design problem.

---
## Section 3 — Data Retrieval from the AlphaFold Database

The AlphaFold DB provides a simple **REST API**.
Given a **UniProt accession identifier**, a GET request returns a JSON payload
containing metadata and a direct URL to the full PDB file for that protein.

```
Step 1 — Fetch metadata:
  GET https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT_ID}
  → JSON array: [{ "pdbUrl": "...", "uniprotDescription": "...", ... }]

Step 2 — Download PDB file:
  GET {pdbUrl}
  → Full PDB text file (pLDDT scores stored in the B-factor column)
```

This approach is instantaneous (AlphaFold DB serves precomputed structures),
reliable (maintained by EMBL-EBI), and coherent with our narrative:
we talk about AlphaFold, and we use AlphaFold's actual outputs.

We fetch structures for two proteins chosen for their **contrasting structural properties**:

| Protein | UniProt ID | Structural character |
|:--------|:----------|:---------------------|
| Human p53 tumour suppressor (TP53) | P04637 | Partially **intrinsically disordered** — the N-terminal transactivation domain has no stable fold in isolation |
| Human lysozyme | P61626 | Small, compact **globular** enzyme — well-structured and highly confident throughout |

The contrast is deliberate: it will allow us to demonstrate both what AlphaFold predicts
with high confidence and how it signals genuine disorder.

### Fallback mechanism

If the AlphaFold API is unreachable (e.g. offline environment),
the fetch function automatically falls back to hardcoded CA-only PDB records
for the first 50 residues of each protein — sufficient to demonstrate all parsing and analysis logic.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fallback PDB data
# CA-only ATOM records for the first 50 residues of each protein.
# Coordinates are placed on a straight line (3.8 Å Cα–Cα spacing).
# pLDDT values are representative of the known structural character of each region.
# ─────────────────────────────────────────────────────────────────────────────

def _make_fallback_pdb(residues, plddt_values):
    '''
    Generate a minimal PDB string with one CA atom per residue.

    Parameters
    ----------
    residues     : list of str   — 3-letter amino acid codes
    plddt_values : list of float — per-residue pLDDT scores (stored as B-factor)

    Returns
    -------
    str — PDB-formatted text with one ATOM record per residue
    '''
    lines = []
    for i, (resname, plddt) in enumerate(zip(residues, plddt_values), start=1):
        x = i * 3.8  # ~3.8 Å between consecutive Cα atoms in an extended chain
        line = (
            f'ATOM  {i:5d}  CA  {resname:3s} A{i:4d}    '
            f'{x:8.3f}   0.000   0.000  1.00{plddt:6.2f}           C  '
        )
        lines.append(line)
    return '\n'.join(lines)


# Human p53 — first 50 residues (UniProt P04637)
# Sequence: N-terminal transactivation domain, intrinsically disordered
_P53_RESIDUES = [
    'MET', 'GLU', 'GLU', 'PRO', 'GLN', 'SER', 'ASP', 'PRO', 'SER', 'VAL',
    'GLU', 'PRO', 'PRO', 'LEU', 'SER', 'GLN', 'GLU', 'THR', 'PHE', 'SER',
    'ASP', 'LEU', 'TRP', 'LYS', 'LEU', 'LEU', 'PRO', 'GLU', 'ASN', 'ASN',
    'VAL', 'LEU', 'SER', 'PRO', 'LEU', 'PRO', 'SER', 'GLN', 'ALA', 'MET',
    'ASP', 'ASP', 'LEU', 'MET', 'LEU', 'SER', 'PRO', 'ASP', 'ASP', 'ILE',
]
_P53_PLDDT = [
    35.42, 32.15, 28.73, 42.18, 38.56, 45.23, 40.11, 35.67, 38.42, 42.89,
    45.12, 40.34, 35.78, 48.23, 45.67, 50.12, 52.34, 48.56, 44.23, 40.78,
    38.34, 42.56, 45.23, 48.67, 42.12, 38.89, 35.45, 40.23, 38.67, 42.34,
    45.78, 40.12, 38.56, 35.23, 40.67, 38.12, 42.89, 45.34, 50.23, 48.67,
    44.12, 42.56, 40.23, 38.78, 40.34, 42.67, 38.12, 35.89, 38.23, 42.56,
]
FALLBACK_P53_PDB = _make_fallback_pdb(_P53_RESIDUES, _P53_PLDDT)


# Human lysozyme — first 50 residues (UniProt P61626)
# Sequence: N-terminal region, well-structured throughout
_LYS_RESIDUES = [
    'LYS', 'VAL', 'PHE', 'GLU', 'ARG', 'CYS', 'GLU', 'LEU', 'ALA', 'ARG',
    'THR', 'LEU', 'LYS', 'ARG', 'LEU', 'GLY', 'MET', 'ASP', 'GLY', 'TYR',
    'ARG', 'GLY', 'ILE', 'SER', 'LEU', 'ALA', 'ASN', 'TRP', 'MET', 'CYS',
    'LEU', 'ALA', 'LYS', 'TRP', 'GLU', 'SER', 'GLY', 'TYR', 'ASN', 'THR',
    'ARG', 'ALA', 'THR', 'ASN', 'TYR', 'ASN', 'ALA', 'GLY', 'ASP', 'ARG',
]
_LYS_PLDDT = [
    88.23, 90.45, 92.67, 94.12, 93.45, 91.23, 92.67, 94.34, 95.12, 93.56,
    91.23, 92.45, 93.67, 94.12, 92.34, 90.56, 91.23, 93.45, 94.67, 95.12,
    93.34, 91.56, 92.23, 93.45, 94.67, 95.12, 93.34, 91.56, 89.23, 90.45,
    92.67, 94.12, 95.34, 93.56, 92.23, 94.45, 95.67, 93.12, 91.34, 92.56,
    94.23, 95.45, 93.67, 91.12, 92.34, 93.56, 94.23, 95.45, 93.67, 91.12,
]
FALLBACK_LYSOZYME_PDB = _make_fallback_pdb(_LYS_RESIDUES, _LYS_PLDDT)


# ─────────────────────────────────────────────────────────────────────────────
# Fetch function
# ─────────────────────────────────────────────────────────────────────────────

def fetch_alphafold_structure(uniprot_id):
    '''
    Retrieve a protein structure from the AlphaFold Protein Structure Database.

    Calls the EBI/DeepMind REST API, downloads the PDB file, and returns the
    content together with protein metadata. Falls back to hardcoded data if
    the API is unreachable.

    Parameters
    ----------
    uniprot_id : str
        UniProt accession identifier (e.g. 'P04637' for human p53).

    Returns
    -------
    dict with keys:
        pdb_string   : str  — full PDB file content as a string
        uniprot_id   : str  — the queried accession identifier
        protein_name : str  — protein description from the UniProt/AlphaFold DB
        organism     : str  — source organism (scientific name)
        is_fallback  : bool — True if hardcoded fallback data was used
    '''
    api_url = f'https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}'

    _fallback_data = {
        'P04637': (FALLBACK_P53_PDB,     'Cellular tumor antigen p53', 'Homo sapiens'),
        'P61626': (FALLBACK_LYSOZYME_PDB, 'Lysozyme C',                'Homo sapiens'),
    }

    try:
        print(f'  Querying API for {uniprot_id}...')
        response = requests.get(api_url, timeout=15)
        response.raise_for_status()
        metadata = response.json()[0]

        pdb_url      = metadata['pdbUrl']
        protein_name = metadata.get('uniprotDescription', 'Unknown')
        organism     = metadata.get('organismScientificName', 'Unknown')

        print(f'  Downloading PDB file from {pdb_url[:60]}...')
        pdb_response = requests.get(pdb_url, timeout=30)
        pdb_response.raise_for_status()
        pdb_string = pdb_response.text

        print(f'  Done. ({len(pdb_string):,} bytes)')
        return {
            'pdb_string':   pdb_string,
            'uniprot_id':   uniprot_id,
            'protein_name': protein_name,
            'organism':     organism,
            'is_fallback':  False,
        }

    except requests.exceptions.RequestException as exc:
        print(f'[WARNING] API call failed for {uniprot_id}: {exc}')
        if uniprot_id in _fallback_data:
            pdb_str, name, org = _fallback_data[uniprot_id]
            print(f'  Using hardcoded fallback data for {name} (50 residues, CA-only).')
            return {
                'pdb_string':   pdb_str,
                'uniprot_id':   uniprot_id,
                'protein_name': name,
                'organism':     org,
                'is_fallback':  True,
            }
        raise RuntimeError(
            f'No fallback available for UniProt ID: {uniprot_id}'
        ) from exc


print('Fallback data and fetch function defined.')

The fetch function is ready. We now call it for both proteins and display a summary table.
The `count_residues` helper counts unique residue sequence numbers in the PDB file
— a quick sanity check that the full-length structure was retrieved.

In [ ]:
print('Fetching structures from the AlphaFold Protein Structure Database...\n')

result_p53      = fetch_alphafold_structure('P04637')
print()
result_lysozyme = fetch_alphafold_structure('P61626')
print()


def count_residues(pdb_string):
    '''
    Count unique residue sequence numbers present in ATOM records of a PDB string.

    Parameters
    ----------
    pdb_string : str — PDB file content

    Returns
    -------
    int — number of unique residue positions
    '''
    seen = set()
    for line in pdb_string.splitlines():
        if line.startswith('ATOM'):
            try:
                seen.add(int(line[22:26]))
            except ValueError:
                pass
    return len(seen)


n_p53      = count_residues(result_p53['pdb_string'])
n_lysozyme = count_residues(result_lysozyme['pdb_string'])

summary_df = pd.DataFrame([
    {
        'Protein':     result_p53['protein_name'],
        'UniProt ID':  result_p53['uniprot_id'],
        'Residues':    n_p53,
        'Organism':    result_p53['organism'],
        'Fallback':    result_p53['is_fallback'],
    },
    {
        'Protein':     result_lysozyme['protein_name'],
        'UniProt ID':  result_lysozyme['uniprot_id'],
        'Residues':    n_lysozyme,
        'Organism':    result_lysozyme['organism'],
        'Fallback':    result_lysozyme['is_fallback'],
    },
])

print('Structures retrieved successfully:\n')
display(summary_df)

---
## Section 4 — PDB Parsing: Extracting pLDDT Scores

### The PDB File Format

The **Protein Data Bank (PDB)** text format has been the standard representation for
macromolecular structures since 1976. It uses fixed-width columns to encode atomic
coordinates and metadata.

The key record type is the **`ATOM` record** — one line per atom:

```
         1         2         3         4         5         6         7
1234567890123456789012345678901234567890123456789012345678901234567890
ATOM      1  CA  MET A   1      14.440   0.000   0.000  1.00 35.42           C
          |  |   |   | |         |       |       |      |    |
        7-11 13-16 18-20 22 23-26    31-38  39-46  47-54 55-60 61-66
       serial name  res  ch resnum    X      Y      Z    occ  B-factor
```

Relevant columns (1-indexed):
- **13–16**: Atom name (`CA` = carbon alpha, the backbone carbon representative of each residue)
- **18–20**: Residue name (3-letter amino acid code)
- **22**: Chain ID
- **23–26**: Residue sequence number
- **31–38 / 39–46 / 47–54**: X, Y, Z coordinates in Angstroms
- **55–60**: Occupancy (normally 1.00)
- **61–66**: **B-factor** (temperature factor)

### How AlphaFold Repurposes the B-factor Column

In crystallographic structures, the **B-factor** measures atomic displacement due to thermal
motion — a measure of how much an atom 'vibrates' in the crystal.

AlphaFold **repurposes this column** to store the **pLDDT score** for each atom.
Since all atoms of a given residue carry the same pLDDT value, we extract one score
per residue by reading the Cα atom only.

This design choice is deliberate: it allows any standard PDB viewer to colour-code
AlphaFold structures by confidence out of the box, using existing B-factor colouring schemes.
We exploit exactly this in the 3D visualisation section.

In Python (0-indexed): `plddt = float(line[60:66])`

In [ ]:
def parse_plddt(pdb_string):
    '''
    Extract per-residue pLDDT scores from an AlphaFold PDB file.

    Reads the B-factor column (columns 61-66 in PDB 1-indexing, i.e. line[60:66])
    of the Calpha (CA) ATOM record for each residue. Returns one row per residue.

    Parameters
    ----------
    pdb_string : str — full PDB file content

    Returns
    -------
    pd.DataFrame with columns:
        residue_index  : int   — 1-based sequential index (row number)
        residue_number : int   — sequence number from the PDB file
        residue_name   : str   — 3-letter amino acid code
        plddt_score    : float — pLDDT confidence value (0-100)
    '''
    records = []

    for line in pdb_string.splitlines():
        if not line.startswith('ATOM'):
            continue
        atom_name = line[12:16].strip()
        if atom_name != 'CA':
            continue
        try:
            residue_number = int(line[22:26])
            residue_name   = line[17:20].strip()
            plddt_score    = float(line[60:66])
        except (ValueError, IndexError):
            continue

        records.append({
            'residue_number': residue_number,
            'residue_name':   residue_name,
            'plddt_score':    plddt_score,
        })

    if not records:
        raise ValueError('No CA ATOM records found in PDB string.')

    df = (
        pd.DataFrame(records)
        .drop_duplicates(subset='residue_number')
        .reset_index(drop=True)
    )
    df.insert(0, 'residue_index', df.index + 1)
    return df


plddt_p53      = parse_plddt(result_p53['pdb_string'])
plddt_lysozyme = parse_plddt(result_lysozyme['pdb_string'])

print(f'p53      — {len(plddt_p53):4d} residues parsed')
print(f'Lysozyme — {len(plddt_lysozyme):4d} residues parsed')
print()
print('First 5 residues of p53 (pLDDT scores reflect the disordered N-terminus):')
display(plddt_p53.head())
print()
print('First 5 residues of lysozyme (note the uniformly high pLDDT):')
display(plddt_lysozyme.head())

---
## Section 5 — Comparative pLDDT Analysis

With per-residue pLDDT scores in hand, we can quantify and visualise the
confidence landscape of each protein. The analysis proceeds in three steps:

1. **Summary statistics** — aggregate confidence distribution for each protein
2. **Profile plot** — per-residue pLDDT along the sequence (the core comparative visualisation)
3. **Disorder detection** — programmatic identification of disordered regions
   (defined as contiguous stretches of ≥10 residues with pLDDT < 70)

### 5a — Summary Statistics

The `compute_disorder_stats` function computes the full confidence distribution
and identifies the longest contiguous low-confidence region in each protein.

In [ ]:
def compute_disorder_stats(plddt_df):
    '''
    Compute summary statistics on per-residue pLDDT scores.

    Parameters
    ----------
    plddt_df : pd.DataFrame — output of parse_plddt, must contain 'plddt_score'

    Returns
    -------
    dict with keys:
        mean_plddt             : float — mean pLDDT across all residues
        pct_very_high          : float — % residues with pLDDT > 90
        pct_confident          : float — % residues with 70 < pLDDT <= 90
        pct_low                : float — % residues with 50 < pLDDT <= 70
        pct_very_low           : float — % residues with pLDDT <= 50
        longest_disorder_len   : int   — length of longest stretch with pLDDT < 70
        longest_disorder_start : int or None — residue_index of that stretch start
        longest_disorder_end   : int or None — residue_index of that stretch end
    '''
    scores = plddt_df['plddt_score'].values
    n_res  = len(scores)

    mean_plddt    = float(np.mean(scores))
    pct_very_high = float(np.sum(scores > 90)                       / n_res * 100)
    pct_confident = float(np.sum((scores > 70) & (scores <= 90))    / n_res * 100)
    pct_low       = float(np.sum((scores > 50) & (scores <= 70))    / n_res * 100)
    pct_very_low  = float(np.sum(scores <= 50)                      / n_res * 100)

    # Find longest contiguous stretch with pLDDT < 70 (likely disordered)
    is_disordered = scores < 70
    best_len, best_start_idx, best_end_idx = 0, 0, 0
    current_start = None

    for idx, disordered in enumerate(is_disordered):
        if disordered and current_start is None:
            current_start = idx
        elif not disordered and current_start is not None:
            run_len = idx - current_start
            if run_len > best_len:
                best_len       = run_len
                best_start_idx = current_start
                best_end_idx   = idx - 1
            current_start = None

    # Handle a disordered stretch that extends to the last residue
    if current_start is not None:
        run_len = n_res - current_start
        if run_len > best_len:
            best_len       = run_len
            best_start_idx = current_start
            best_end_idx   = n_res - 1

    residue_indices = plddt_df['residue_index'].values
    start_res = int(residue_indices[best_start_idx]) if best_len > 0 else None
    end_res   = int(residue_indices[best_end_idx])   if best_len > 0 else None

    return {
        'mean_plddt':             round(mean_plddt, 1),
        'pct_very_high':          round(pct_very_high, 1),
        'pct_confident':          round(pct_confident, 1),
        'pct_low':                round(pct_low, 1),
        'pct_very_low':           round(pct_very_low, 1),
        'longest_disorder_len':   best_len,
        'longest_disorder_start': start_res,
        'longest_disorder_end':   end_res,
    }


stats_p53      = compute_disorder_stats(plddt_p53)
stats_lysozyme = compute_disorder_stats(plddt_lysozyme)

# Print formatted comparison table
col_w = 36
header = f'{"Metric":<{col_w}} {"p53 (P04637)":>18} {"Lysozyme (P61626)":>20}'
sep    = '-' * len(header)

rows = [
    ('Mean pLDDT',                      stats_p53['mean_plddt'],          stats_lysozyme['mean_plddt']),
    ('% Very high confidence  (>90)',   stats_p53['pct_very_high'],       stats_lysozyme['pct_very_high']),
    ('% Confident             (70-90)', stats_p53['pct_confident'],       stats_lysozyme['pct_confident']),
    ('% Low confidence        (50-70)', stats_p53['pct_low'],             stats_lysozyme['pct_low']),
    ('% Very low / disordered  (<50)',  stats_p53['pct_very_low'],        stats_lysozyme['pct_very_low']),
    ('Longest disordered region (res)', stats_p53['longest_disorder_len'],stats_lysozyme['longest_disorder_len']),
]

print(header)
print(sep)
for label, val_p53, val_lys in rows:
    print(f'{label:<{col_w}} {str(val_p53):>18} {str(val_lys):>20}')

print()
if stats_p53['longest_disorder_len'] > 0:
    print(f'  p53 longest disordered region : '
          f'residues {stats_p53["longest_disorder_start"]}-{stats_p53["longest_disorder_end"]} '
          f'({stats_p53["longest_disorder_len"]} residues)')
if stats_lysozyme['longest_disorder_len'] > 0:
    print(f'  Lysozyme longest disordered region : '
          f'residues {stats_lysozyme["longest_disorder_start"]}-{stats_lysozyme["longest_disorder_end"]} '
          f'({stats_lysozyme["longest_disorder_len"]} residues)')
else:
    print('  Lysozyme: no disordered region detected (all residues above threshold).')

### 5b — Per-residue pLDDT Profile Plot

The following plot shows the pLDDT score for every residue, from N-terminus (left) to C-terminus (right).
Background bands encode the confidence tiers defined in Section 1.
The longest disordered region detected in p53 is annotated with an arrow.

Both subplots share the same y-axis to make the contrast immediately visible.

In [ ]:
fig, (ax_p53, ax_lys) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# ── Shared background bands (same for both panels) ──────────────────────────
for ax in (ax_p53, ax_lys):
    ax.axhspan(0,   50, alpha=0.15, color='red',       zorder=0)
    ax.axhspan(50,  70, alpha=0.15, color='orange',    zorder=0)
    ax.axhspan(70,  90, alpha=0.15, color='lightblue', zorder=0)
    ax.axhspan(90, 100, alpha=0.15, color='royalblue', zorder=0)
    for thresh in (50, 70, 90):
        ax.axhline(thresh, color='grey', linewidth=0.8, linestyle='--', alpha=0.6, zorder=1)

# ── Left panel: p53 ─────────────────────────────────────────────────────────
x_p53    = plddt_p53['residue_index'].values
sc_p53   = plddt_p53['plddt_score'].values
xmax_p53 = int(x_p53[-1])

ax_p53.plot(x_p53, sc_p53, color='steelblue', linewidth=1.2, zorder=2)
ax_p53.set_title(result_p53['protein_name'], fontsize=13, fontweight='bold')
ax_p53.set_xlabel('Residue index', fontsize=11)
ax_p53.set_ylabel('pLDDT score', fontsize=11)
ax_p53.set_ylim(0, 100)
ax_p53.set_xlim(1, xmax_p53)

# Annotate the longest disordered region
if stats_p53['longest_disorder_len'] > 0:
    mid_x  = (stats_p53['longest_disorder_start'] + stats_p53['longest_disorder_end']) / 2.0
    text_x = min(mid_x + max(55, xmax_p53 * 0.14), xmax_p53 * 0.82)
    ax_p53.annotate(
        'Intrinsically\ndisordered domain',
        xy=(mid_x, 42),
        xytext=(text_x, 18),
        arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
        fontsize=9,
        color='darkred',
        ha='center',
    )

# ── Right panel: lysozyme ────────────────────────────────────────────────────
x_lys    = plddt_lysozyme['residue_index'].values
sc_lys   = plddt_lysozyme['plddt_score'].values
xmax_lys = int(x_lys[-1])

ax_lys.plot(x_lys, sc_lys, color='steelblue', linewidth=1.2, zorder=2)
ax_lys.set_title(result_lysozyme['protein_name'], fontsize=13, fontweight='bold')
ax_lys.set_xlabel('Residue index', fontsize=11)
ax_lys.set_ylim(0, 100)
ax_lys.set_xlim(1, xmax_lys)

# ── Shared legend (placed on the right panel) ────────────────────────────────
legend_patches = [
    mpatches.Patch(facecolor='royalblue', alpha=0.6, label='Very high (>90)'),
    mpatches.Patch(facecolor='lightblue', alpha=0.6, label='Confident (70-90)'),
    mpatches.Patch(facecolor='orange',    alpha=0.6, label='Low (50-70)'),
    mpatches.Patch(facecolor='red',       alpha=0.6, label='Very low (<50)'),
]
ax_lys.legend(handles=legend_patches, loc='lower right', fontsize=8, framealpha=0.85)

fig.suptitle('Per-residue pLDDT confidence profiles', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plddt_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as plddt_comparison.png')

**Figure 1 — Per-residue pLDDT confidence profiles for p53 (left) and lysozyme (right).**

*p53 (P04637):* The N-terminal transactivation domain (roughly residues 1–100)
shows pLDDT scores consistently below 50 — flagged by the model as very low confidence.
This is not a prediction failure: this region is **genuinely intrinsically disordered in solution**.
It only acquires a defined conformation upon binding a partner protein (e.g. the MDM2 oncoprotein
or transcriptional co-activators such as CBP/p300).
The DNA-binding domain (residues ~100–290) shows high pLDDT (typically > 80),
reflecting its compact, well-folded beta-sandwich structure coordinated around a zinc ion.

*Lysozyme (P61626):* Confidence is uniformly high throughout (mean pLDDT > 90).
This is expected: lysozyme is a small globular enzyme with a tight hydrophobic core,
four disulfide bridges, and an extraordinarily stable fold studied since the 1960s.
It is precisely the type of protein that AlphaFold models with near-experimental accuracy.

**Key observation**: low pLDDT is *meaningful information*, not noise.
It encodes the model's own uncertainty, which correlates with genuine conformational flexibility.
AlphaFold signals when structure prediction is not the right frame.

### 5c — Disorder Detection

We now algorithmically identify **disordered regions**: contiguous stretches of at least
10 consecutive residues with pLDDT < 70.
This threshold corresponds to the boundary between 'confident' and 'low confidence'
in AlphaFold's own scale, and is widely used in the literature to define predicted disorder.

In [ ]:
def find_disordered_regions(plddt_df, min_length=10, threshold=70.0):
    '''
    Identify contiguous disordered regions in a pLDDT profile.

    A region is considered disordered if pLDDT < threshold for at least
    min_length consecutive residues.

    Parameters
    ----------
    plddt_df   : pd.DataFrame — output of parse_plddt
    min_length : int          — minimum contiguous length to report (default 10)
    threshold  : float        — pLDDT cutoff for disorder (default 70.0)

    Returns
    -------
    pd.DataFrame with columns:
        Region, Start residue, End residue, Length, Mean pLDDT
    '''
    regions       = []
    region_idx    = 1
    current_start = None
    region_scores = []

    for _, row in plddt_df.iterrows():
        if row['plddt_score'] < threshold:
            if current_start is None:
                current_start = int(row['residue_index'])
            region_scores.append(row['plddt_score'])
        else:
            if current_start is not None:
                length = len(region_scores)
                if length >= min_length:
                    regions.append({
                        'Region':        region_idx,
                        'Start residue': current_start,
                        'End residue':   current_start + length - 1,
                        'Length':        length,
                        'Mean pLDDT':    round(float(np.mean(region_scores)), 1),
                    })
                    region_idx += 1
                current_start = None
                region_scores = []

    # Handle a disordered stretch extending to the last residue
    if current_start is not None and len(region_scores) >= min_length:
        length = len(region_scores)
        regions.append({
            'Region':        region_idx,
            'Start residue': current_start,
            'End residue':   current_start + length - 1,
            'Length':        length,
            'Mean pLDDT':    round(float(np.mean(region_scores)), 1),
        })

    empty_schema = ['Region', 'Start residue', 'End residue', 'Length', 'Mean pLDDT']
    return pd.DataFrame(regions) if regions else pd.DataFrame(columns=empty_schema)


disorder_p53      = find_disordered_regions(plddt_p53)
disorder_lysozyme = find_disordered_regions(plddt_lysozyme)

print('Disordered regions in p53 (P04637)')
print('Criterion: >= 10 consecutive residues with pLDDT < 70\n')
if disorder_p53.empty:
    print('  None detected.')
else:
    display(disorder_p53)

print()
print('Disordered regions in Lysozyme (P61626)')
print('Criterion: >= 10 consecutive residues with pLDDT < 70\n')
if disorder_lysozyme.empty:
    print('  None detected — lysozyme is uniformly well-structured.')
else:
    display(disorder_lysozyme)

---
## Section 6 — 3D Structure Visualisation

The pLDDT profile gives a one-dimensional view of confidence along the sequence.
The 3D structure shows what that confidence landscape looks like in space.

We use **py3Dmol**, a JavaScript-based molecular viewer embedded directly in Jupyter,
to render each protein as an interactive 3D model.
Both structures are displayed in **cartoon** representation
(a simplified ribbon showing secondary structure: helices as ribbons, strands as arrows)
and **coloured by pLDDT** via the B-factor column:

| Colour | Confidence |
|:-------|:-----------|
| Blue   | High pLDDT (well-structured) |
| White  | Intermediate pLDDT |
| Red    | Low pLDDT (likely disordered) |

**Prediction**: the 3D view should confirm what the pLDDT plot showed.
p53's N-terminal region should appear as a red, elongated, loosely-structured chain
without compact secondary structure.
Lysozyme should be almost entirely blue — a dense, well-folded globular shape.

In [ ]:
# ── 3D visualisation: Human p53 ──────────────────────────────────────────────
# Colour scheme: blue = high pLDDT, white = medium, red = low pLDDT
# (The 'rwb' gradient maps B-factor minimum to red, maximum to blue.)

print('Human p53 tumour suppressor — P04637')
print('Colour: blue = high pLDDT | white = medium | red = low pLDDT')
print('(Interact: left-drag to rotate, scroll to zoom, right-drag to translate)\n')

view_p53 = py3Dmol.view(width=720, height=520)
view_p53.addModel(result_p53['pdb_string'], 'pdb')
view_p53.setStyle(
    {},
    {
        'cartoon': {
            'colorscheme': {
                'prop':     'b',
                'gradient': 'rwb',
                'min':      0,
                'max':      100,
            }
        }
    },
)
view_p53.zoomTo()
view_p53.show()

**p53 (P04637):** The elongated red/orange region visible at one end is the
**N-terminal transactivation domain** — intrinsically disordered, with no stable compact fold.
The blue compact core is the **DNA-binding domain** (residues ~100–290),
the primary druggable region of p53, targeted by cancer therapies such as
APR-246 (Eprenetapopt), which rescues mutant p53 by restoring wild-type-like folding.
The tetramerisation domain (residues ~320–360) may also appear as a compact blue region.

In [ ]:
# ── 3D visualisation: Human lysozyme ─────────────────────────────────────────
# Same colour scheme as p53 for direct visual comparison.

print('Human lysozyme — P61626')
print('Colour: blue = high pLDDT | white = medium | red = low pLDDT')
print('(Interact: left-drag to rotate, scroll to zoom, right-drag to translate)\n')

view_lys = py3Dmol.view(width=720, height=520)
view_lys.addModel(result_lysozyme['pdb_string'], 'pdb')
view_lys.setStyle(
    {},
    {
        'cartoon': {
            'colorscheme': {
                'prop':     'b',
                'gradient': 'rwb',
                'min':      0,
                'max':      100,
            }
        }
    },
)
view_lys.zoomTo()
view_lys.show()

**Lysozyme (P61626):** The structure is almost uniformly blue — AlphaFold is highly confident
about every residue position. The compact globular fold, with its characteristic alpha helices
and beta sheet, matches the crystal structure first determined by David Phillips in 1965
(one of the first enzyme structures ever solved, predating the PDB itself).
The near-perfect pLDDT reflects the protein's dense hydrophobic core and four disulfide bridges,
which strongly constrain the native conformation — leaving little structural ambiguity for the model.

---
## Section 7 — Interpretation and Biological Insight

### What the Comparison Demonstrates

The side-by-side analysis of p53 and lysozyme is not arbitrary.
These two proteins sit at opposite ends of a structural spectrum:

**Lysozyme** is the prototypical compact globular protein.
Its 148 residues form a well-defined hydrophobic core, held rigid by four disulfide bridges.
It has been crystallised thousands of times under dozens of conditions,
and its structure is essentially invariant.
AlphaFold predicts it with near-perfect confidence (mean pLDDT > 90) and
no disordered regions are detected — exactly what the model is designed for.

**p53** presents a more complex picture.
The DNA-binding domain (residues ~100–290) shows high pLDDT (typically > 80):
this region forms a compact beta-sandwich that directly contacts DNA,
stabilised by a coordinated zinc ion and a network of hydrophobic interactions.
AlphaFold models it reliably, and the structure aligns closely with experimental crystal structures.

The N-terminal **transactivation domain** (residues ~1–100), however, scores below 50.
This is biologically correct: this domain is **intrinsically disordered protein (IDP)** in solution.
It exists as a dynamic ensemble of conformations, and only adopts a defined local structure
upon binding a specific partner — MDM2, the TATA-binding protein, CBP/p300, and others.
No single stable conformation exists in isolation, so there is no structure to predict.

### Low pLDDT Does Not Mean Model Failure

This is the most important conceptual point of this section:

> *AlphaFold does not fail on p53's N-terminal domain.
> It correctly signals that this region has no single stable structure.
> The model encodes genuine uncertainty as a low pLDDT score,
> which correlates strongly with experimentally verified intrinsically disordered regions (IDRs).*

A model that confidently predicted a spurious fold for an IDR would be **less** useful —
it would mislead researchers into targeting a phantom structure.
AlphaFold's self-calibrated uncertainty is a scientifically sophisticated and practically valuable output.

### Connection to Drug Discovery

The DNA-binding domain of p53 (high pLDDT, well-structured) is precisely the domain
exploited by cancer therapies. Approximately **50% of human cancers** carry a TP53 mutation,
many of which destabilise this domain.

Knowing its atomic-resolution structure enables:
- **Structure-based drug design**: design small molecules that stabilise the wild-type fold
  or rescue destabilised mutants
- **Resistance mechanism analysis**: understand which mutations affect the active site or
  allosteric pockets
- **Fragment-based lead discovery**: identify fragments binding to known druggable pockets
  for iterative optimisation

A concrete example: **APR-246 (Eprenetapopt)**, in clinical trials for TP53-mutant
myelodysplastic syndrome, converts to a Michael acceptor that covalently modifies cysteine
residues in mutant p53, restoring wild-type-like folding and transcriptional activity.
This mechanism was elucidated through structural analysis of the DNA-binding domain.

The **AlphaFold DB** is freely accessible at https://alphafold.ebi.ac.uk.
Researchers worldwide query it daily to identify druggable pockets,
compare orthologous structures across species, and prioritise experimental follow-up.

---
## Section 8 — Towards Molecular Interaction: Graph Neural Networks

We now know the three-dimensional structure of our target protein at atomic resolution —
including which regions are reliably folded and where the druggable binding sites are located.
The next challenge is finding a **small molecule** whose shape and chemical properties
complement that binding pocket.

Evaluating whether a candidate molecule will be soluble, chemically stable, non-toxic,
and biologically active against our target requires reasoning over the molecule's own structure:
its atoms, bonds, chirality, and electronic properties.
Small molecules are naturally represented as **graphs** — atoms as nodes, bonds as edges.

**Graph Neural Networks (GNNs)** are a class of deep learning models designed to operate
directly on graph-structured data, propagating information between neighbouring nodes
over multiple message-passing layers to learn molecular representations.
They are the current method of choice for predicting molecular properties from chemical structure,
and the subject of the next notebook.